# Bài 2 – Phân loại hình tròn, vuông và tam giác

Notebook kiểm tra dữ liệu hình học, minh họa tiền xử lý, huấn luyện 9 cấu hình Shape/HOG kết hợp KNN/SVM/Random Forest và đánh giá trên ảnh mới.

In [ ]:
from pathlib import Path
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'tuan6').exists() and (ROOT.parent / 'tuan6').exists():
    ROOT = ROOT.parent
assert (ROOT / 'tuan6').exists(), 'Hãy chạy notebook từ thư mục gốc của repository.'
DATA = ROOT / 'tuan6' / 'data' / 'shapes'
DATA_NEW = ROOT / 'tuan6' / 'data' / 'shapes_external'
OUTPUT = ROOT / 'tuan6' / 'output_bai2'
ROOT

## 1. Kiểm tra dữ liệu

In [ ]:
classes = {'Tròn': 'circle', 'Vuông': 'square', 'Tam giác': 'triangle'}
counts = {name: len(list((DATA / folder).glob('*.png'))) for name, folder in classes.items()}
pd.Series(counts, name='Số ảnh').to_frame()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, folder) in zip(axes, classes.items()):
    path = next((DATA / folder).glob('*.png'))
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()

## 2. Tùy chọn: sinh lại dữ liệu

Dữ liệu đã được lưu trong repository. Chỉ chạy cell dưới nếu muốn tạo lại bằng seed mặc định.

In [ ]:
# subprocess.run([sys.executable, str(ROOT / 'tuan6' / 'tao_du_lieu_hinh_hoc.py')], check=True)

## 3. Huấn luyện và đánh giá

Ảnh được chuyển xám, làm mờ Gaussian, phân ngưỡng Otsu, tìm biên Canny và contour. Chương trình trích 7 đặc trưng hình dạng, HOG 1.764 chiều và vector kết hợp 1.771 chiều.

In [ ]:
command = [
    sys.executable, str(ROOT / 'tuan6' / 'bai2_phan_loai_hinh_dang.py'),
    '--data', str(DATA), '--data-moi', str(DATA_NEW),
    '--output', str(OUTPUT), '--seed', '42'
]
subprocess.run(command, check=True)

## 4. Bảng kết quả

In [ ]:
shape_features = pd.read_csv(OUTPUT / '02_bang_dac_trung_hinh_dang.csv')
hog_features = np.load(OUTPUT / '03_dac_trung_hog.npy')
metrics = pd.read_csv(OUTPUT / '04_bang_so_sanh_mo_hinh.csv')
print('Shape:', shape_features.shape, '| HOG:', hog_features.shape)
display(metrics)

In [ ]:
result_images = [
    ('Tiền xử lý và HOG', '01_minh_hoa_tien_xu_ly_hog.png'),
    ('So sánh 9 thí nghiệm', '05_bieu_do_so_sanh.png'),
    ('Ma trận nhầm lẫn', '06_confusion_matrix.png'),
    ('Dự đoán 15 ảnh mới', '08_du_doan_anh_moi.png'),
    ('Phân tích mẫu sai', '09_phan_tich_mau_sai.png'),
]
for title, filename in result_images:
    image = plt.imread(OUTPUT / filename)
    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

## 5. Nhận xét

Cả 9 cấu hình đạt Accuracy và F1-macro bằng 1,00 trên tập kiểm thử. Đặc trưng Shape chỉ có 7 chiều nhưng đã phân biệt tốt ba hình cơ bản, nên nhẹ hơn HOG. HOG hữu ích khi biên và cấu trúc ảnh phức tạp hơn. Kết quả cao là hợp lý với dữ liệu sinh tổng hợp sạch; ảnh thực tế cần được kiểm thử riêng dưới thay đổi ánh sáng, nền và biến dạng.